# 14 Message Passing Across Agent Nodes

Notebook generado a partir del paquete Python. Los módulos se incluyen en orden de dependencia (los módulos importados por otros aparecen primero).

> Nota: los `import` de módulos locales del paquete original se conservan tal cual. Como todos los módulos están combinados en este notebook en orden de dependencia, los símbolos referenciados ya quedan definidos en celdas anteriores.

## ¿Qué hace este notebook?

Demuestra el **paso de mensajes entre nodos-agente**. El flujo es
`generator → reviewer → refiner`: el generador produce un borrador, el revisor aporta
crítica y el refinador entrega la versión final. Cada agente añade su mensaje (con su
`name`) al estado compartido, de modo que el siguiente nodo construye sobre lo anterior.

El notebook une, en orden de dependencia, `llm_provider` (`ChatAnthropic`),
`message_passing_graph` (grafo) y `main` (entrada de tarea + impresión de la traza).

## Ejemplo de uso

**Datos de interacción que espera el agente.** Cadena fija `generator → reviewer → refiner`
sin pausa; concluye en un `invoke`.

- Entrada inicial esperada: `{"messages": [HumanMessage(content="<tarea>")]}`.
- Cada nodo añade su mensaje (con `name`) al estado, que el siguiente nodo recibe; la salida
  contiene la traza completa.

```python
from langchain_core.messages import HumanMessage

llm = get_llm()
app = build_graph(llm)
final_state = app.invoke(
    {"messages": [HumanMessage(content="Escribe una función para validar emails")]}
)                                                   # generator→reviewer→refiner
for i, m in enumerate(final_state["messages"], 1):
    name = getattr(m, "name", None) or "user/system"
    print(f"{i:02d}. [{name.upper()}] {m.content}\n")
```

In [1]:
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic

In [2]:

# Load environment variables from .env file
load_dotenv()

True

In [3]:


def get_llm():
    # Read Anthropic API key
    api_key = os.getenv("ANTHROPIC_API_KEY")

    # Read Claude model name (default if not set)
    model = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

    # Ensure API key is available
    if not api_key:
        raise ValueError("ANTHROPIC_API_KEY is missing in .env")

    # Return configured Claude LLM
    return ChatAnthropic(
        model=model,
        api_key=api_key,
        temperature=0,
    )

In [4]:
from typing import TypedDict

from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langgraph.graph import StateGraph, MessagesState, START, END

In [5]:


class State(MessagesState):
    # Shared state across nodes
    draft: str
    feedback: str

In [6]:


def generator_node(llm):
    def _node(state: State) -> dict:
        print("→ Generator node: creating a first draft...")

        # Get latest user task
        task = state["messages"][-1].content

        # System prompt for generator
        sys = SystemMessage(
            content=(
                "Role: Generator\n"
                "Create a concise first draft based on the user's task.\n"
                "Output: a short, structured draft.\n"
            )
        )

        # Generate draft
        resp = llm.invoke([sys, HumanMessage(content=task)])
        draft = resp.content.strip()

        # Append generator output to messages
        return {
            "draft": draft,
            "messages": [HumanMessage(content=draft, name="generator")]
        }

    return _node

In [7]:


def reviewer_node(llm):
    def _node(state: State) -> dict:
        print("→ Reviewer node: providing feedback on the draft...")

        # System prompt for reviewer
        sys = SystemMessage(
            content=(
                "Role: Reviewer\n"
                "Review the draft and provide actionable feedback.\n"
                "Focus on clarity, completeness, and correctness.\n"
                "Output: bullet points.\n"
            )
        )

        # Get draft from state
        draft = state.get("draft", "")

        # Generate feedback
        resp = llm.invoke([sys, HumanMessage(content=draft)])
        feedback = resp.content.strip()

        # Append reviewer feedback to messages
        return {
            "feedback": feedback,
            "messages": [HumanMessage(content=feedback, name="reviewer")]
        }

    return _node

In [8]:


def refiner_node(llm):
    def _node(state: State) -> dict:
        print("→ Refiner node: producing the improved final version...")

        # System prompt for refiner
        sys = SystemMessage(
            content=(
                "Role: Refiner\n"
                "Revise the draft using the reviewer feedback.\n"
                "Output: final polished version.\n"
            )
        )

        # Read draft and feedback from state
        draft = state.get("draft", "")
        feedback = state.get("feedback", "")

        # Build refinement prompt
        prompt = (
            "Draft:\n"
            f"{draft}\n\n"
            "Feedback:\n"
            f"{feedback}\n\n"
            "Return the revised final answer:"
        )

        # Generate refined output
        resp = llm.invoke([sys, HumanMessage(content=prompt)])
        final_answer = resp.content.strip()

        # Append final output to messages
        return {
            "messages": [HumanMessage(content=final_answer, name="refiner")]
        }

    return _node

In [9]:


def build_graph(llm):
    # Create stateful graph
    graph = StateGraph(State)

    # Register graph nodes
    graph.add_node("generator", generator_node(llm))
    graph.add_node("reviewer", reviewer_node(llm))
    graph.add_node("refiner", refiner_node(llm))

    # Define linear execution flow
    graph.add_edge(START, "generator")
    graph.add_edge("generator", "reviewer")
    graph.add_edge("reviewer", "refiner")
    graph.add_edge("refiner", END)

    return graph.compile()

In [10]:
from langchain_core.messages import HumanMessage


In [11]:


def banner():
    # Print header
    print("\n" + "=" * 60)
    print(" Message Passing Across Agent Nodes (LangGraph + Claude)")
    print("=" * 60)
    print("Flow: generator → reviewer → refiner\n")

In [12]:


def print_trace(messages):
    # Print messages passed between nodes
    print("\n--- Message Trace ---\n")
    for i, m in enumerate(messages, start=1):
        name = getattr(m, "name", None) or "user/system"
        print(f"{i:02d}. [{name.upper()}]")
        print(m.content)
        print()

In [13]:


def main():
    banner()

    # Initialize LLM and LangGraph app
    llm = get_llm()
    app = build_graph(llm)

    # Read user task
    task = input("Enter a task (or exit()): ").strip()
    if task.lower() in {"exit()", "exit", "quit"}:
        print("\nExiting.\n")
        return
    if not task:
        print("Please provide a valid task.\n")
        return

    print("\n--- Graph Execution Started ---\n")

    # Invoke the graph with the initial user message
    final_state = app.invoke({"messages": [HumanMessage(content=task)]})

    # Display full message history
    print_trace(final_state["messages"])

    # Display additional state stored by the graph
    print("--- State Artifacts ---\n")
    print("DRAFT (stored in state['draft']):\n")
    print(final_state.get("draft", ""))
    print("\nFEEDBACK (stored in state['feedback']):\n")
    print(final_state.get("feedback", ""))
    print("\n" + "-" * 60 + "\n")

In [14]:


if __name__ == "__main__":
    main()


 Message Passing Across Agent Nodes (LangGraph + Claude)
Flow: generator → reviewer → refiner


--- Graph Execution Started ---

→ Generator node: creating a first draft...
→ Reviewer node: providing feedback on the draft...
→ Refiner node: producing the improved final version...

--- Message Trace ---

01. [USER/SYSTEM]
Develop fibonacci function

02. [GENERATOR]
# Fibonacci Function

## Multiple Implementations

### 1. Recursive (Simple but slow)
```python
def fibonacci_recursive(n):
    """Returns the nth Fibonacci number using recursion."""
    if n <= 0:
        return 0
    elif n == 1:
        return 1
    return fibonacci_recursive(n - 1) + fibonacci_recursive(n - 2)
```

---

### 2. Iterative (Efficient)
```python
def fibonacci_iterative(n):
    """Returns the nth Fibonacci number using iteration."""
    if n <= 0:
        return 0
    elif n == 1:
        return 1

    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b
```

---

### 3. Memoized / 